# M5 TFT — Kiểm chứng độ dài cửa sổ encoder

Notebook độc lập, nhúng bản chụp feature/model/loss từ `tft_m5_zero_test_improved.ipynb`.
Không cần notebook gốc hoặc các file Python đi kèm khi chạy trên Kaggle.

**Câu hỏi:** với bộ feature hiện tại, encoder 14/28/56/84/112 ngày có giúp dự báo tốt hơn 56 ngày không?
Horizon cố định 28 ngày; cùng origins, batch và feature bank. Chỉ đọc sales đến d_1913.

## Cách chạy
- Dependencies: numpy, pandas, torch, matplotlib. Kaggle GPU được khuyến nghị cho full M5.
- Đặt `M5_DATA_ROOT` nếu cần; tự tìm Kaggle input và data/raw.
- Mặc định full: 30.490 series, 3 seeds, tối đa 20 epoch, 160 origins, batch 256.
- Kiểm tra nhanh: đặt `WINDOW_SMOKE=1`; chỉ dùng để kiểm tra pipeline.
- Ví dụ trong cell riêng trước config: `%env WINDOW_SMOKE=1` và `%env WINDOW_LENGTHS=28,56`.
- Chạy seed đầu trước: `%env WINDOW_RUN_SEEDS=42`, giữ WINDOW_SEEDS mặc định để khóa grid 3 seed.
- Sau đó xóa WINDOW_RUN_SEEDS hoặc đặt `42,123,2026`: các seed/lượt hoàn tất được resume, chỉ chạy lượt thiếu.
- Thay đổi grid WINDOW_SEEDS hoặc WINDOW_LENGTHS cần OUT_DIR mới do manifest khóa cấu hình.
- Output: `WINDOW_OUT_DIR` (mặc định `window_experiment_outputs`).
- `WINDOW_CONFIRM_CUTOFFS=1801,1829` bật các origin kiểm tra độ nhạy; mỗi cutoff có inner validation riêng.

**Protocol:** inner train đến d_1857 → chọn epoch trên d_1858–1885 → khởi tạo lại/refit đến d_1885 → chấm outer validation d_1886–1913.
Metric chính: WRMSSE-5 BU chưa calibration. Không dùng outer validation để early stopping.
WRMSSE-5 không phải WRMSSE 12 cấp official M5.


In [ ]:
import os
import sys
import json
import math
import random
import time
import gc
import hashlib
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SMOKE = os.getenv('WINDOW_SMOKE', '0') == '1'
WINDOWS = sorted(set(int(x) for x in os.getenv('WINDOW_LENGTHS', '14,28,56,84,112').split(',')))
SEEDS = [int(x) for x in os.getenv('WINDOW_SEEDS', '42' if SMOKE else '42,123,2026').split(',')]
RUN_SEEDS = [int(x) for x in os.getenv('WINDOW_RUN_SEEDS', ','.join(map(str,SEEDS))).split(',')]
assert 56 in WINDOWS and min(WINDOWS) > 0 and len(SEEDS) == len(set(SEEDS))
assert RUN_SEEDS and set(RUN_SEEDS).issubset(SEEDS) and len(RUN_SEEDS)==len(set(RUN_SEEDS))
MAX_SEQ = max(WINDOWS)
N_SERIES = int(os.getenv('WINDOW_N_SERIES', '32' if SMOKE else '30490'))
MAX_EPOCHS = int(os.getenv('WINDOW_MAX_EPOCHS', '1' if SMOKE else '20'))
MAX_TRAIN_ORIGINS = int(os.getenv('WINDOW_MAX_ORIGINS', '2' if SMOKE else '160'))
BATCH = int(os.getenv('WINDOW_BATCH', '16' if SMOKE else '256'))
D_MODEL = int(os.getenv('WINDOW_D_MODEL', '16' if SMOKE else '64'))
HEADS = int(os.getenv('WINDOW_HEADS', '2' if SMOKE else '4'))
PATIENCE = int(os.getenv('WINDOW_PATIENCE', '3'))
H = 28
INNER_END, INNER_VAL_END = 1857, 1885
OUTER_END = 1913
# Optional additional refit cutoffs: e.g. 1801,1829; each forecasts next 28 days.
CONFIRM_CUTOFFS = sorted(set(int(x) for x in os.getenv('WINDOW_CONFIRM_CUTOFFS', '').split(',') if x))
assert all(MAX_SEQ + 2*H <= c < INNER_END and c + H <= OUTER_END for c in CONFIRM_CUTOFFS)
assert MAX_SEQ + H <= INNER_END
assert min(N_SERIES, MAX_EPOCHS, BATCH, D_MODEL, HEADS, PATIENCE) > 0 and D_MODEL % HEADS == 0
SEQ, SEED = 56, 42  # fixed data-selection seed, independent of training seeds
TRAIN_END, VAL_START, VAL_END = INNER_END, INNER_END+1, INNER_VAL_END
TEST_START, TEST_END = INNER_VAL_END+1, OUTER_END  # compatibility names; this is OUTER VALIDATION
CALIBRATION_DAYS = 14
SALES_FILE = 'sales_train_validation.csv'  # only d_1..d_1913; never competition test targets
PROFILE = 'window_experiment'
DROPOUT, LR, WEIGHT_DECAY = .1, .001, .0001
LOSS, TWEEDIE_P = 'tweedie', 1.3
SERIES_WEIGHT_POWER = .5
HIERARCHICAL_LOSS_WEIGHT, ZERO_LOSS_WEIGHT = .2, .1
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
NUM_WORKERS = 0
OUT_DIR = Path(os.getenv('WINDOW_OUT_DIR', 'window_experiment_outputs'))
OUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = OUT_DIR / 'runs'
RUN_DIR.mkdir(exist_ok=True)

def safe_str(value):
    return str(value).encode('ascii', 'backslashreplace').decode('ascii')

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
print('Device:', DEVICE, '| windows:', WINDOWS, '| seeds:', SEEDS, '| smoke:', SMOKE)
print('Outer validation: d_1886-d_1913. Competition test d_1914-d_1941 is not read.')



## 1. Helpers nhúng độc lập
Các module evaluator và zero-aware features được tạo trong bộ nhớ, không ghi file helper ra thư mục chạy.

In [ ]:
import types
module = types.ModuleType('_window_wrmsse5')
exec('"""Leakage-safe WRMSSE evaluation for M5 levels L1/L2/L3/L9/L12."""\n\nfrom __future__ import annotations\n\nfrom typing import Mapping\n\nimport numpy as np\n\n\nLEVELS = ("L1", "L2", "L3", "L9", "L12")\n\n\ndef group_sum(values: np.ndarray, groups: np.ndarray, n_groups: int | None = None) -> np.ndarray:\n    values = np.asarray(values, dtype=np.float64)\n    groups = np.asarray(groups, dtype=np.int64)\n    if values.ndim != 2:\n        raise ValueError("values must have shape [series, time]")\n    if groups.ndim != 1 or len(groups) != len(values):\n        raise ValueError("groups must contain one code per series")\n    if n_groups is None:\n        n_groups = int(groups.max()) + 1\n    out = np.zeros((n_groups, values.shape[1]), dtype=np.float64)\n    np.add.at(out, groups, values)\n    return out\n\n\ndef rmsse_scale(y_history: np.ndarray) -> np.ndarray:\n    """Return M5 RMSSE denominators after removing pre-launch leading zeros."""\n    y_history = np.asarray(y_history, dtype=np.float64)\n    if y_history.ndim != 2:\n        raise ValueError("y_history must have shape [series, time]")\n\n    scales = np.full(len(y_history), np.nan, dtype=np.float64)\n    for row_index, row in enumerate(y_history):\n        nonzero = np.flatnonzero(row != 0)\n        if not len(nonzero):\n            continue\n        active = row[nonzero[0]:]\n        if len(active) < 2:\n            continue\n        scales[row_index] = np.mean(np.diff(active) ** 2)\n    return scales\n\n\ndef build_contexts(\n    groups_by_level: Mapping[str, np.ndarray],\n    sales_history: np.ndarray,\n    price_history: np.ndarray,\n    weight_window: int = 28,\n) -> dict[str, dict[str, np.ndarray | int | str]]:\n    """Precompute scale and pre-forecast dollar weights for the five levels."""\n    sales_history = np.asarray(sales_history)\n    price_history = np.asarray(price_history)\n    if sales_history.shape != price_history.shape:\n        raise ValueError("sales_history and price_history must have the same shape")\n    if sales_history.ndim != 2 or sales_history.shape[1] == 0:\n        raise ValueError("history must have shape [series, positive_time]")\n    if weight_window <= 0:\n        raise ValueError("weight_window must be positive")\n\n    start = max(0, sales_history.shape[1] - weight_window)\n    recent_sales = sales_history[:, start:].astype(np.float64)\n    recent_price = np.nan_to_num(price_history[:, start:], nan=0.0).astype(np.float64)\n    series_revenue = np.sum(recent_sales * recent_price, axis=1)\n    contexts = {}\n    for level in LEVELS:\n        groups = np.asarray(groups_by_level[level], dtype=np.int64)\n        n_groups = int(groups.max()) + 1\n        grouped_history = group_sum(sales_history, groups, n_groups)\n        scale = rmsse_scale(grouped_history)\n        revenue = np.bincount(groups, weights=series_revenue, minlength=n_groups).astype(np.float64)\n        valid = np.isfinite(scale) & (scale > 1e-12)\n        weights = np.where(valid, np.maximum(revenue, 0.0), 0.0)\n        weight_sum = weights.sum()\n        if weight_sum > 0:\n            weights /= weight_sum\n        elif valid.any():\n            weights[valid] = 1.0 / valid.sum()\n\n        contexts[level] = {\n            "level": level,\n            "groups": groups,\n            "n_groups": n_groups,\n            "scale": scale,\n            "weights": weights,\n            "valid": valid,\n        }\n    return contexts\n\n\ndef evaluate_level(\n    y_true: np.ndarray,\n    y_pred: np.ndarray,\n    context: Mapping[str, np.ndarray | int | str],\n) -> dict[str, float | int | str]:\n    """Evaluate one hierarchy level using a pre-forecast evaluation context."""\n    y_true = np.asarray(y_true, dtype=np.float64)\n    y_pred = np.asarray(y_pred, dtype=np.float64)\n    if y_true.shape != y_pred.shape:\n        raise ValueError("y_true and y_pred must have the same shape")\n\n    groups = np.asarray(context["groups"], dtype=np.int64)\n    n_groups = int(context["n_groups"])\n    scale = np.asarray(context["scale"], dtype=np.float64)\n    weights = np.asarray(context["weights"], dtype=np.float64)\n    valid = np.asarray(context["valid"], dtype=bool)\n    yt = group_sum(y_true, groups, n_groups)\n    yp = group_sum(y_pred, groups, n_groups)\n\n    mse = np.mean((yt - yp) ** 2, axis=1)\n    rmsse = np.full(n_groups, np.nan, dtype=np.float64)\n    rmsse[valid] = np.sqrt(mse[valid] / scale[valid])\n    wrmsse = float(np.sum(weights[valid] * rmsse[valid]))\n\n    error = np.abs(yt - yp)\n    actual_by_group = np.abs(yt).sum(axis=1)\n    group_ok = actual_by_group > 1e-12\n    group_bias = np.full(n_groups, np.nan, dtype=np.float64)\n    group_bias[group_ok] = (yp - yt).sum(axis=1)[group_ok] / actual_by_group[group_ok]\n    positive = np.abs(yt) > 0\n\n    return {\n        "level": str(context["level"]),\n        "n_keys": n_groups,\n        "n_valid_scales": int(valid.sum()),\n        "n_obs": int(yt.size),\n        "WRMSSE": wrmsse,\n        "RMSE": float(np.sqrt(np.mean((yt - yp) ** 2))),\n        "MAE": float(error.mean()),\n        "WMAPE": float(error.sum() / max(np.abs(yt).sum(), 1e-12)),\n        "MAPE": float(np.mean(error[positive] / np.abs(yt[positive]))) if positive.any() else np.nan,\n        "Bias": float((yp - yt).sum() / max(np.abs(yt).sum(), 1e-12)),\n        "Ratio": float(yp.sum() / max(yt.sum(), 1e-12)),\n        "MeanAbsGroupBias": float(np.nanmean(np.abs(group_bias))) if group_ok.any() else np.nan,\n        "P90AbsGroupBias": float(np.nanpercentile(np.abs(group_bias), 90)) if group_ok.any() else np.nan,\n    }\n\n\ndef evaluate_wrmsse5(\n    y_true: np.ndarray,\n    y_pred: np.ndarray,\n    contexts: Mapping[str, Mapping[str, np.ndarray | int | str]],\n) -> tuple[list[dict[str, float | int | str]], float]:\n    rows = [evaluate_level(y_true, y_pred, contexts[level]) for level in LEVELS]\n    return rows, float(np.mean([float(row["WRMSSE"]) for row in rows]))\n', module.__dict__)
sys.modules['_window_wrmsse5'] = module
module = types.ModuleType('_window_hurdle_features')
exec('"""Leakage-safe local and hierarchical features for the M5 HurdleTFT."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Mapping\n\nimport numpy as np\n\n\nLOCAL_STATE_COLS = (\n    "nonzero_rate_7",\n    "nonzero_rate_28",\n    "nonzero_rate_56",\n    "log_zero_run",\n    "has_ever_sold",\n    "log_age_since_first_sale",\n    "log_mean_positive_28",\n    "listed_rate_28",\n    "log_last_positive_units",\n)\n\nHIER_CHANNELS = (\n    "log_sales",\n    "log_roll_mean_7",\n    "log_roll_mean_28",\n    "trend_7_28",\n    "listed_ratio",\n)\n\n\ndef rolling_sum(values: np.ndarray, window: int) -> np.ndarray:\n    """Inclusive causal rolling sum along time for a two-dimensional panel."""\n    values = np.asarray(values, dtype=np.float32)\n    if values.ndim != 2:\n        raise ValueError("values must have shape [series, time]")\n    if window <= 0:\n        raise ValueError("window must be positive")\n\n    cumulative = np.cumsum(values, axis=1, dtype=np.float32)\n    result = cumulative.copy()\n    if window < values.shape[1]:\n        result[:, window:] -= cumulative[:, :-window]\n    return result\n\n\ndef rolling_mean(values: np.ndarray, window: int) -> np.ndarray:\n    result = rolling_sum(values, window)\n    counts = np.minimum(np.arange(1, values.shape[1] + 1), window).astype(np.float32)\n    result /= counts[None, :]\n    return result\n\n\ndef build_local_state_features(\n    sales: np.ndarray,\n    price_available: np.ndarray,\n) -> np.ndarray:\n    """Build L12 occurrence, lifecycle and past-listing features.\n\n    Every value at day ``t`` depends only on observations at or before ``t``.\n    Float16 storage keeps the full 30,490-series panel practical; windows are\n    converted back to float32 by the dataset.\n    """\n    sales = np.asarray(sales, dtype=np.float32)\n    price_available = np.asarray(price_available, dtype=bool)\n    if sales.ndim != 2 or price_available.shape != sales.shape:\n        raise ValueError("sales and price_available must share shape [series, time]")\n    if np.any(sales < 0):\n        raise ValueError("sales must be non-negative")\n\n    n_series, n_days = sales.shape\n    output = np.empty((n_series, n_days, len(LOCAL_STATE_COLS)), dtype=np.float16)\n    positive = sales > 0\n    positive_float = positive.astype(np.float32)\n\n    for name, window in (\n        ("nonzero_rate_7", 7),\n        ("nonzero_rate_28", 28),\n        ("nonzero_rate_56", 56),\n    ):\n        output[:, :, LOCAL_STATE_COLS.index(name)] = rolling_mean(\n            positive_float, window\n        ).astype(np.float16)\n\n    run = np.zeros(n_series, dtype=np.int16)\n    first_sale = np.full(n_series, -1, dtype=np.int16)\n    last_positive = np.zeros(n_series, dtype=np.float32)\n    zero_run_idx = LOCAL_STATE_COLS.index("log_zero_run")\n    ever_idx = LOCAL_STATE_COLS.index("has_ever_sold")\n    age_idx = LOCAL_STATE_COLS.index("log_age_since_first_sale")\n    last_positive_idx = LOCAL_STATE_COLS.index("log_last_positive_units")\n    for day in range(n_days):\n        sold = positive[:, day]\n        first_sale[(first_sale < 0) & sold] = day\n        run = np.where(sold, 0, np.minimum(run + 1, 364)).astype(np.int16)\n        last_positive = np.where(sold, sales[:, day], last_positive)\n        seen = first_sale >= 0\n        age = np.where(seen, day - first_sale + 1, 0)\n        output[:, day, zero_run_idx] = np.log1p(run).astype(np.float16)\n        output[:, day, ever_idx] = seen.astype(np.float16)\n        output[:, day, age_idx] = np.log1p(age).astype(np.float16)\n        output[:, day, last_positive_idx] = np.log1p(last_positive).astype(np.float16)\n\n    positive_count_28 = rolling_sum(positive_float, 28)\n    positive_sales_28 = rolling_sum(sales, 28)\n    mean_positive_28 = np.divide(\n        positive_sales_28,\n        positive_count_28,\n        out=np.zeros_like(positive_sales_28),\n        where=positive_count_28 > 0,\n    )\n    output[:, :, LOCAL_STATE_COLS.index("log_mean_positive_28")] = np.log1p(\n        mean_positive_28\n    ).astype(np.float16)\n    output[:, :, LOCAL_STATE_COLS.index("listed_rate_28")] = rolling_mean(\n        price_available.astype(np.float32), 28\n    ).astype(np.float16)\n    return output\n\n\ndef group_sum(values: np.ndarray, groups: np.ndarray) -> np.ndarray:\n    values = np.asarray(values, dtype=np.float32)\n    groups = np.asarray(groups, dtype=np.int64)\n    if values.ndim != 2 or groups.ndim != 1 or len(groups) != len(values):\n        raise ValueError("invalid values/groups shape")\n    if len(groups) == 0 or np.any(groups < 0):\n        raise ValueError("groups must contain non-negative codes")\n    result = np.zeros((int(groups.max()) + 1, values.shape[1]), dtype=np.float32)\n    np.add.at(result, groups, values)\n    return result\n\n\ndef build_hierarchy_context(\n    sales: np.ndarray,\n    price_available: np.ndarray,\n    groups_by_level: Mapping[str, np.ndarray],\n    levels: tuple[str, ...] = ("L1", "L2", "L3", "L9"),\n) -> dict[str, np.ndarray]:\n    """Build compact parent histories without broadcasting them to L12."""\n    sales = np.asarray(sales, dtype=np.float32)\n    price_available = np.asarray(price_available, dtype=bool)\n    if sales.ndim != 2 or price_available.shape != sales.shape:\n        raise ValueError("sales and price_available must share shape [series, time]")\n\n    contexts = {}\n    for level in levels:\n        groups = np.asarray(groups_by_level[level], dtype=np.int64)\n        aggregate = group_sum(sales, groups)\n        roll_7 = rolling_mean(aggregate, 7)\n        roll_28 = rolling_mean(aggregate, 28)\n        trend = np.clip((roll_7 - roll_28) / (roll_28 + 1.0), -5.0, 5.0)\n        listed_count = group_sum(price_available.astype(np.float32), groups)\n        group_size = np.bincount(groups, minlength=aggregate.shape[0]).astype(np.float32)\n        listed_ratio = listed_count / np.maximum(group_size[:, None], 1.0)\n        contexts[level] = np.stack(\n            [\n                np.log1p(aggregate),\n                np.log1p(roll_7),\n                np.log1p(roll_28),\n                trend,\n                listed_ratio,\n            ],\n            axis=-1,\n        ).astype(np.float32)\n    return contexts\n\n\ndef fit_feature_scaler(values: np.ndarray, train_end: int) -> tuple[np.ndarray, np.ndarray]:\n    """Fit feature-wise moments strictly before the one-based train cutoff."""\n    values = np.asarray(values)\n    if values.ndim < 2 or not 0 < train_end <= values.shape[1]:\n        raise ValueError("invalid train_end")\n    axes = tuple(range(values.ndim - 1))\n    train = values[:, :train_end].astype(np.float32)\n    mean = train.mean(axis=axes, dtype=np.float64).astype(np.float32)\n    std = train.std(axis=axes, dtype=np.float64).astype(np.float32)\n    std[std < 1e-6] = 1.0\n    return mean, std\n\n\ndef hierarchy_feature_names(\n    levels: tuple[str, ...] = ("L1", "L2", "L3", "L9"),\n) -> list[str]:\n    return [f"{level}_{channel}" for level in levels for channel in HIER_CHANNELS]\n', module.__dict__)
sys.modules['_window_hurdle_features'] = module
from _window_wrmsse5 import LEVELS, build_contexts, evaluate_wrmsse5
from _window_hurdle_features import HIER_CHANNELS, LOCAL_STATE_COLS, build_hierarchy_context, build_local_state_features, fit_feature_scaler, hierarchy_feature_names
SOURCE_HASH = '43b84cbeb15939e535ddddb1a0fe15f5e7f13ed6b0a0857ebbeb79cef28073e8'


## 2. Sales, calendar, giá và feature bank
Giữ nguyên cách tạo feature TFT improved. Chỉ nạp d_1–1913; không nạp test d_1914–1941. Selection seed cố định 42. Subset aggregates không phải tổng M5 đầy đủ.

In [ ]:
def find_data_root():
    candidates = []
    if os.environ.get("M5_DATA_ROOT"):
        configured = Path(os.environ["M5_DATA_ROOT"]).expanduser()
        candidates.append(configured.parent if configured.is_file() else configured)
    candidates += [
        Path("/kaggle/input/m5-forecasting-accuracy"),
        Path("../data/raw"),
        Path("data/raw"),
        Path("../../data/raw"),
    ]
    for path in candidates:
        if (path / SALES_FILE).exists():
            return path.resolve()

    # Kaggle dataset slugs and uploaded dataset layouts can add extra folders.
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        matches = sorted(kaggle_input.rglob(SALES_FILE))
        complete = [
            match.parent
            for match in matches
            if (match.parent / "calendar.csv").exists()
            and (match.parent / "sell_prices.csv").exists()
        ]
        if complete:
            return complete[0].resolve()

        attached = sorted(path.name for path in kaggle_input.iterdir())
        attached_text = ", ".join(attached) if attached else "(khong co dataset nao)"
    else:
        attached_text = "/kaggle/input khong ton tai"
    raise FileNotFoundError(
        f"Khong tim thay {SALES_FILE} kem calendar.csv va sell_prices.csv. "
        "Tren Kaggle, chon Add Input > M5 Forecasting Accuracy, sau do Run All lai. "
        f"Hoac dat M5_DATA_ROOT toi thu muc chua ba file. Input hien co: {attached_text}"
    )


DATA_ROOT = find_data_root()
print("data_root:", safe_str(DATA_ROOT))


def select_series(sales, n_series, seed=42):
    if n_series >= len(sales):
        return sales.reset_index(drop=True)

    stores = sorted(sales["store_id"].unique())
    per_store = max(1, int(math.ceil(n_series / len(stores))))
    parts = []

    for store in stores:
        g = sales[sales["store_id"].eq(store)].copy()
        first = g.sort_values(["dept_id", "item_id"]).groupby("dept_id", sort=True).head(1)
        rest = g.drop(first.index).sample(frac=1.0, random_state=seed)
        picked = pd.concat([first, rest]).drop_duplicates("id").head(per_store)
        parts.append(picked)

    out = pd.concat(parts).drop_duplicates("id")
    if len(out) < n_series:
        rest = sales[~sales["id"].isin(out["id"])]
        if len(rest):
            out = pd.concat([out, rest.sample(n=min(n_series - len(out), len(rest)), random_state=seed)])
    return out.sample(frac=1.0, random_state=seed).reset_index(drop=True)


sales_all = pd.read_csv(DATA_ROOT / SALES_FILE, dtype={"id": str}, usecols=lambda c: not c.startswith("d_") or int(c[2:]) <= OUTER_END)
meta_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
day_cols = sorted([c for c in sales_all.columns if c.startswith("d_")], key=lambda x: int(x[2:]))
n_days = len(day_cols)

assert n_days >= TEST_END, f"Can it nhat {TEST_END} ngay, dang co {n_days}"
if PROFILE == "full_5level" and len(sales_all) != 30490:
    raise ValueError(
        f"PROFILE=full_5level can dung 30.490 series, nhung {SALES_FILE} chi co "
        f"{len(sales_all):,}. Kiem tra lai M5_DATA_ROOT va dataset Kaggle."
    )
sales_sel = select_series(sales_all, N_SERIES, SEED)
series_ids = sales_sel["id"].tolist()
n_series = len(series_ids)
if PROFILE == "full_5level" and n_series != 30490:
    raise ValueError(
        f"Full profile phai chon 30.490 series, hien chi chon {n_series:,}."
    )

print("series selected:", n_series)
print("days:", n_days)
print(sales_sel.groupby(["store_id", "state_id"]).size().to_string())



In [ ]:
SALES_DF = sales_sel.set_index("id").loc[series_ids, day_cols]
SALES_DF.columns = np.arange(1, n_days + 1)
SALES_DF = SALES_DF.astype(np.float32).fillna(0.0)
SALES = SALES_DF.to_numpy(dtype=np.float32)

cal = pd.read_csv(DATA_ROOT / "calendar.csv")
cal["d_num"] = cal["d"].str[2:].astype(np.int16)
cal["event_name_any"] = cal["event_name_1"].fillna(cal["event_name_2"])
cal["event_type"] = cal["event_type_1"].fillna(cal["event_type_2"]).fillna("None")
cal["has_event"] = cal["event_name_any"].notna().astype(np.float32)
cal["event_type_code"] = cal["event_type"].astype("category").cat.codes.astype(np.float32) + 1.0
cal = cal.sort_values("d_num").set_index("d_num")
cal = cal.reindex(np.arange(1, n_days + 1)).reset_index()
cal["wday"] = cal["wday"].fillna(0).astype(np.float32)
cal["month"] = cal["month"].fillna(0).astype(np.float32)
cal["year"] = cal["year"].fillna(0).astype(np.float32)
cal["wm_yr_wk"] = cal["wm_yr_wk"].fillna(0).astype(np.int64)
cal["snap_CA"] = cal["snap_CA"].fillna(0).astype(np.float32)
cal["snap_TX"] = cal["snap_TX"].fillna(0).astype(np.float32)
cal["snap_WI"] = cal["snap_WI"].fillna(0).astype(np.float32)
cal["has_event"] = cal["has_event"].fillna(0).astype(np.float32)
cal["event_type_code"] = cal["event_type_code"].fillna(0).astype(np.float32)

CAL = {
    "is_weekend": cal["wday"].isin([1, 2]).to_numpy(np.float32),
    "sin_wday": np.sin(2.0 * np.pi * cal["wday"].to_numpy(np.float32) / 7.0).astype(np.float32),
    "cos_wday": np.cos(2.0 * np.pi * cal["wday"].to_numpy(np.float32) / 7.0).astype(np.float32),
    "sin_month": np.sin(2.0 * np.pi * cal["month"].to_numpy(np.float32) / 12.0).astype(np.float32),
    "cos_month": np.cos(2.0 * np.pi * cal["month"].to_numpy(np.float32) / 12.0).astype(np.float32),
    "sin_year": np.sin(2.0 * np.pi * cal["d_num"].to_numpy(np.float32) / 365.25).astype(np.float32),
    "cos_year": np.cos(2.0 * np.pi * cal["d_num"].to_numpy(np.float32) / 365.25).astype(np.float32),
    "has_event": cal["has_event"].to_numpy(np.float32),
    "event_type_code": cal["event_type_code"].to_numpy(np.float32),
}

SNAP = np.zeros((n_series, n_days), dtype=np.float32)
for state, col in [("CA", "snap_CA"), ("TX", "snap_TX"), ("WI", "snap_WI")]:
    mask = sales_sel["state_id"].eq(state).to_numpy()
    if mask.any():
        SNAP[mask] = cal[col].to_numpy(np.float32)[None, :]

prices = pd.read_csv(
    DATA_ROOT / "sell_prices.csv",
    usecols=["store_id", "item_id", "wm_yr_wk", "sell_price"],
)
pairs = sales_sel[["store_id", "item_id"]].drop_duplicates()
prices = prices.merge(pairs, on=["store_id", "item_id"], how="inner")

# Expand weekly prices directly to days. Avoiding a 59M-row melt/merge keeps the
# full 30,490-series profile within Kaggle RAM.
price_weekly = prices.pivot_table(
    index=["store_id", "item_id"],
    columns="wm_yr_wk",
    values="sell_price",
    aggfunc="last",
)
pair_order = sales_sel.set_index("id").loc[series_ids, ["store_id", "item_id"]]
pair_index = pd.MultiIndex.from_frame(pair_order)
price_weekly = price_weekly.reindex(pair_index)
week_positions = pd.Index(price_weekly.columns).get_indexer(cal["wm_yr_wk"].to_numpy())
price_values = np.full((n_series, n_days), np.nan, dtype=np.float32)
valid_week = week_positions >= 0
price_values[:, valid_week] = price_weekly.to_numpy(np.float32)[:, week_positions[valid_week]]
PRICE_RAW = pd.DataFrame(price_values, index=series_ids, columns=np.arange(1, n_days + 1))
PRICE_OBS = PRICE_RAW.to_numpy(np.float32)
PRICE_AVAILABLE = np.isfinite(PRICE_OBS)

# Price is unknown in the forecast horizon. Past prices are filled causally;
# pre-listing days stay at zero and are identified by is_listed.
PRICE_DF = PRICE_RAW.ffill(axis=1).fillna(0.0)
PRICE = PRICE_DF.to_numpy(np.float32)

price_median_28 = (
    PRICE_DF.shift(1, axis=1).T.rolling(28, min_periods=1).median().T
)
PRICE_REL = (PRICE_DF / price_median_28.replace(0.0, np.nan))
PRICE_REL = PRICE_REL.replace([np.inf, -np.inf], np.nan).fillna(1.0).clip(0.0, 5.0)
PRICE_REL = PRICE_REL.to_numpy(np.float32)
PROMO = (PRICE_REL < 0.95).astype(np.float32)

print("sales matrix:", SALES.shape)
print("price matrix:", PRICE.shape)
print("past price availability rate:", float(PRICE_AVAILABLE.mean()))
print("zero rate:", float((SALES == 0).mean()))
print("promo rate:", float(PROMO.mean()))

del prices, pairs, price_weekly, price_values, pair_order, pair_index, cal, sales_all



In [ ]:
PAST_COLS = [
    "sales", "lag_1", "lag_7", "lag_14", "lag_28", "lag_35", "lag_56", "lag_364",
    "roll_mean_7", "roll_mean_28", "roll_mean_56", "roll_std_28",
    "sell_price", "price_rel", "promo", "is_listed", "snap", "has_event",
    "sin_wday", "cos_wday", "sin_month", "cos_month",
]
FUT_COLS = [
    "is_weekend", "sin_wday", "cos_wday", "sin_month", "cos_month",
    "sin_year", "cos_year", "snap", "has_event", "event_type_code",
]
FEATURE_COLS = list(dict.fromkeys(PAST_COLS + FUT_COLS))

FEAT = np.zeros((n_series, n_days, len(FEATURE_COLS)), dtype=np.float32)


def store_feature(name, values):
    FEAT[:, :, FEATURE_COLS.index(name)] = values


# Write one feature at a time instead of retaining ~9 GB of intermediate maps.
S = SALES_DF
store_feature("sales", SALES)
for k in [1, 7, 14, 28, 35, 56, 364]:
    store_feature(f"lag_{k}", S.shift(k, axis=1).fillna(0.0).to_numpy(np.float32))

for window in [7, 28, 56]:
    store_feature(
        f"roll_mean_{window}",
        S.T.rolling(window, min_periods=1).mean().T.to_numpy(np.float32),
    )
store_feature(
    "roll_std_28",
    S.T.rolling(28, min_periods=2).std().T.fillna(0.0).to_numpy(np.float32),
)

store_feature("sell_price", PRICE)
store_feature("price_rel", PRICE_REL)
store_feature("promo", PROMO)
store_feature("is_listed", PRICE_AVAILABLE.astype(np.float32))
store_feature("snap", SNAP)
for name in [
    "is_weekend", "has_event", "event_type_code", "sin_wday", "cos_wday",
    "sin_month", "cos_month", "sin_year", "cos_year",
]:
    store_feature(name, CAL[name][None, :])

PAST_IDX = [FEATURE_COLS.index(c) for c in PAST_COLS]
FUT_IDX = [FEATURE_COLS.index(c) for c in FUT_COLS]

STATIC_COLS = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
STATIC_DF = sales_sel.set_index("id").loc[series_ids, STATIC_COLS]
STATIC = np.zeros((n_series, len(STATIC_COLS)), dtype=np.int64)
STATIC_CARD = []
for j, col in enumerate(STATIC_COLS):
    codes, uniques = pd.factorize(STATIC_DF[col].astype(str), sort=True)
    STATIC[:, j] = codes + 1
    STATIC_CARD.append(len(uniques) + 1)


def dense_codes(codes):
    uniques = np.unique(codes)
    mapping = {value: i for i, value in enumerate(uniques)}
    return np.array([mapping[v] for v in codes], dtype=np.int64)


store_code = STATIC[:, STATIC_COLS.index("store_id")]
dept_code = STATIC[:, STATIC_COLS.index("dept_id")]
state_code = STATIC[:, STATIC_COLS.index("state_id")]
GROUPS = {
    "L1": np.zeros(n_series, dtype=np.int64),
    "L2": dense_codes(state_code),
    "L3": dense_codes(store_code),
    "L9": dense_codes(store_code * 1000 + dept_code),
    "L12": np.arange(n_series, dtype=np.int64),
}
if PROFILE == "full_5level":
    actual_group_counts = {level: int(GROUPS[level].max()) + 1 for level in LEVELS}
    expected_group_counts = {"L1": 1, "L2": 3, "L3": 10, "L9": 70, "L12": 30490}
    if actual_group_counts != expected_group_counts:
        raise ValueError(
            f"Hierarchy full khong dung: {actual_group_counts}; can {expected_group_counts}"
        )
HIERARCHY_GROUP_TENSORS = {
    level: torch.as_tensor(GROUPS[level], dtype=torch.long, device=DEVICE)
    for level in ("L1", "L2", "L3", "L9")
}

HIER_LEVELS = ("L1", "L2", "L3", "L9")
LOCAL_STATE = build_local_state_features(SALES, PRICE_AVAILABLE)
HIER_CONTEXT = build_hierarchy_context(SALES, PRICE_AVAILABLE, GROUPS, HIER_LEVELS)
HIER_CONTEXT_COLS = hierarchy_feature_names(HIER_LEVELS)
MODEL_PAST_COLS = PAST_COLS + list(LOCAL_STATE_COLS) + HIER_CONTEXT_COLS

VAL_EVAL_CONTEXTS = build_contexts(
    GROUPS,
    SALES[:, :VAL_START - 1],
    PRICE_OBS[:, :VAL_START - 1],
)
TEST_EVAL_CONTEXTS = build_contexts(
    GROUPS,
    SALES[:, :TEST_START - 1],
    PRICE_OBS[:, :TEST_START - 1],
)
CAL_SELECT_EVAL_CONTEXTS = build_contexts(
    GROUPS,
    SALES[:, :VAL_START - 1 + CALIBRATION_DAYS],
    PRICE_OBS[:, :VAL_START - 1 + CALIBRATION_DAYS],
)

del S
del SALES_DF, PRICE_DF, PRICE_RAW, price_median_28

print(
    "features:", FEAT.shape,
    "| past base/local/hierarchy:", len(PAST_COLS), len(LOCAL_STATE_COLS), len(HIER_CONTEXT_COLS),
    "| future:", len(FUT_COLS),
)
print("static cards:", STATIC_CARD)
print({k: int(v.max()) + 1 for k, v in GROUPS.items()})



## 3. Model và loss
Cùng TFTLite occurrence/severity heads, Tweedie + zero loss + hierarchical auxiliary loss. Không calibration. Bỏ phần tạo model/trainer tự động của notebook gốc.

In [ ]:
class GRN(nn.Module):

    def __init__(self, d_model, ctx_dim=0, dropout=0.1):
        super().__init__()
        in_dim = d_model + ctx_dim
        self.fc1 = nn.Linear(in_dim, 2 * d_model)
        self.fc2 = nn.Linear(2 * d_model, d_model)
        self.gate = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, ctx=None):
        h = torch.cat([x, ctx], dim=-1) if ctx is not None else x
        h = F.elu(self.fc1(h))
        h = self.drop(h)
        h = self.fc2(h)
        h = self.drop(h)
        gate = torch.sigmoid(self.gate(h))
        return self.norm(x + gate * h)

class VSN(nn.Module):

    def __init__(self, n_features, d_model, dropout=0.1):
        super().__init__()
        self.weight_net = nn.Sequential(nn.Linear(d_model, d_model), nn.ELU(), nn.Linear(d_model, n_features))
        self.proj = nn.Linear(n_features, d_model)
        self.grn = GRN(d_model, dropout=dropout)

    def forward(self, x, ctx):
        w = torch.softmax(self.weight_net(ctx), dim=-1)
        return self.grn(self.proj(x * w.unsqueeze(1)))

class StaticEmbedding(nn.Module):

    def __init__(self, cards, d_model):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(card, d_model) for card in cards])
        self.proj = nn.Linear(d_model * len(cards), d_model)

    def forward(self, static):
        parts = [emb(static[:, i]) for i, emb in enumerate(self.embeddings)]
        return self.proj(torch.cat(parts, dim=-1))

class TFTLite(nn.Module):

    def __init__(self, n_past, n_future, cards, d_model=32, heads=2, dropout=0.1, horizon=28):
        super().__init__()
        self.horizon = horizon
        self.static = StaticEmbedding(cards, d_model)
        self.past_vsn = VSN(n_past, d_model, dropout)
        self.fut_vsn = VSN(n_future, d_model, dropout)
        self.encoder = nn.LSTM(d_model, d_model, batch_first=True)
        self.decoder = nn.LSTM(d_model, d_model, batch_first=True)
        self.static_ctx = nn.Linear(d_model, d_model)
        self.enrich = GRN(d_model, ctx_dim=d_model, dropout=dropout)
        self.attn = nn.MultiheadAttention(d_model, heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(d_model)
        self.ff = GRN(d_model, dropout=dropout)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(d_model, 1)
        self.zero_head = nn.Linear(d_model, 1)

    def forward(self, past, future, static, return_zero_logits=False):
        ctx = self.static(static)
        past = self.past_vsn(past, ctx)
        future = self.fut_vsn(future, ctx)
        enc, (h, c) = self.encoder(past)
        dec, _ = self.decoder(future, (h, c))
        x = torch.cat([enc, dec], dim=1)
        x = x + self.static_ctx(ctx).unsqueeze(1)
        x = self.enrich(x, ctx.unsqueeze(1).expand(-1, x.size(1), -1))
        length = x.size(1)
        causal = torch.triu(torch.ones(length, length, device=x.device, dtype=torch.bool), diagonal=1)
        attn, _ = self.attn(x, x, x, attn_mask=causal, need_weights=False)
        x = self.attn_norm(x + self.drop(attn))
        x = self.ff(x)
        forecast_state = x[:, -self.horizon:, :]
        severity = F.softplus(self.head(forecast_state).squeeze(-1)) + 0.0001
        zero_logits = self.zero_head(forecast_state).squeeze(-1)
        mean = torch.sigmoid(zero_logits) * severity
        if return_zero_logits:
            return (mean, zero_logits)
        return mean

In [ ]:
def tweedie_loss_elementwise(y, mu, p=TWEEDIE_P):
    mu = mu.clamp_min(1e-06)
    return -y * torch.pow(mu, 1.0 - p) / (1.0 - p) + torch.pow(mu, 2.0 - p) / (2.0 - p)

def masked_loss(y, mu, mask):
    if LOSS == 'tweedie':
        loss = tweedie_loss_elementwise(y, mu)
    else:
        loss = (mu - y) ** 2
    return (loss * mask).sum() / mask.sum().clamp_min(1.0)

def zero_occurrence_loss(y, logits, mask):
    occurrence = (y > 0).to(logits.dtype)
    loss = F.binary_cross_entropy_with_logits(logits, occurrence, reduction='none')
    return (loss * mask).sum() / mask.sum().clamp_min(1.0)

def hierarchical_aux_loss(y, mu, mask, series_index):
    """Relative aggregate error for L1/L2/L3/L9 within an origin-aligned batch."""
    valid = (mask > 0).float()
    pred = mu.float() * valid
    actual = y.float() * valid
    losses = []
    for level in ('L1', 'L2', 'L3', 'L9'):
        codes = HIERARCHY_GROUP_TENSORS[level][series_index]
        _, inverse = torch.unique(codes, sorted=False, return_inverse=True)
        n_groups = int(inverse.max().item()) + 1
        pred_sum = torch.zeros(n_groups, pred.size(1), device=pred.device, dtype=pred.dtype)
        actual_sum = torch.zeros_like(pred_sum)
        count = torch.zeros_like(pred_sum)
        pred_sum.index_add_(0, inverse, pred)
        actual_sum.index_add_(0, inverse, actual)
        count.index_add_(0, inverse, valid)
        group_valid = count > 0
        denominator = actual_sum.detach().abs() + count.clamp_min(1.0)
        relative_error = (pred_sum - actual_sum) / denominator
        losses.append((relative_error[group_valid] ** 2).mean())
    return torch.stack(losses).mean()

def eval_all(y_true, y_pred, contexts, tag):
    rows, overall = evaluate_wrmsse5(y_true, y_pred, contexts)
    df = pd.DataFrame(rows)
    df.insert(0, 'model', tag)
    return (df, overall)

def make_summary_row(frame, tag, overall):
    l1 = frame[frame['level'].eq('L1')].iloc[0]
    l12 = frame[frame['level'].eq('L12')].iloc[0]
    row = {'model': tag, 'WRMSSE_5levels': overall, 'L1_WMAPE': l1['WMAPE'], 'L12_WMAPE': l12['WMAPE'], 'L12_Bias': l12['Bias'], 'L12_Ratio': l12['Ratio']}
    for level in LEVELS:
        row[f'{level}_WRMSSE'] = frame.loc[frame['level'].eq(level), 'WRMSSE'].iloc[0]
    return row

def seasonal_naive(sales, target_start, lag, horizon=28):
    history_end = target_start - 1
    if history_end < lag:
        raise ValueError(f'Need at least {lag} history days')
    pattern = sales[:, history_end - lag:history_end]
    repeats = int(math.ceil(horizon / lag))
    return np.tile(pattern, (1, repeats))[:, :horizon].astype(np.float32)

def weekday_mean_naive(sales, target_start, horizon=28, weeks=4):
    history_end = target_start - 1
    window = 7 * weeks
    if history_end < window:
        raise ValueError(f'Need at least {window} history days')
    recent = sales[:, history_end - window:history_end].reshape(len(sales), weeks, 7)
    weekday_mean = recent.mean(axis=1)
    return np.tile(weekday_mean, (1, int(math.ceil(horizon / 7))))[:, :horizon].astype(np.float32)

def zero_metrics(y_true, y_pred, threshold=0.5):
    actual = y_true.ravel()
    pred = y_pred.ravel()
    actual_zero = actual < 0.5
    pred_zero = pred < threshold
    tp = float((actual_zero & pred_zero).sum())
    fp = float((~actual_zero & pred_zero).sum())
    fn = float((actual_zero & ~pred_zero).sum())
    precision = tp / max(tp + fp, 1.0)
    recall = tp / max(tp + fn, 1.0)
    f1 = 2.0 * precision * recall / max(precision + recall, 1e-09)
    nonzero = ~actual_zero
    return {'zero_precision': precision, 'zero_recall': recall, 'zero_f1': f1, 'zero_rate_actual': float(actual_zero.mean()), 'zero_rate_pred': float(pred_zero.mean()), 'mae_nonzero': float(np.abs(actual[nonzero] - pred[nonzero]).mean()) if nonzero.any() else np.nan, 'bias_nonzero': float((pred[nonzero] - actual[nonzero]).mean()) if nonzero.any() else np.nan, 'bias_zero': float((pred[actual_zero] - actual[actual_zero]).mean()) if actual_zero.any() else np.nan}

def fit_store_factors(pred, true, groups, low=0.8, high=1.2):
    factors = {}
    for g in np.unique(groups):
        mask = groups == g
        denominator = float(pred[mask].sum())
        factor = 1.0 if denominator <= 1e-09 else float(true[mask].sum() / denominator)
        factors[g] = float(np.clip(factor, low, high))
    return factors

def apply_factors(pred, groups, factors):
    out = pred.copy()
    for g, factor in factors.items():
        out[groups == g] *= factor
    return out

def fit_series_factors(pred, true, k=5.0, low=0.5, high=2.0):
    pred_sum = pred.sum(axis=1).astype(np.float64)
    true_sum = true.sum(axis=1).astype(np.float64)
    factors = (true_sum + k) / np.maximum(pred_sum + k, 1e-09)
    factors = np.where(pred_sum + true_sum > 0, factors, 1.0)
    return np.clip(factors, low, high).astype(np.float32)

def apply_series_factors(pred, factors):
    return pred * factors[:, None]

def run_epoch(model, loader, optimizer=None, scaler=None, scheduler=None):
    train = optimizer is not None
    model.train(train)
    total_loss = 0.0
    total_base_loss = 0.0
    total_hier_loss = 0.0
    total_zero_loss = 0.0
    total_rows = 0
    for past, future, static, target, mask, series_index, _ in loader:
        past = past.to(DEVICE, non_blocking=True)
        future = future.to(DEVICE, non_blocking=True)
        static = static.to(DEVICE, non_blocking=True)
        target = target.to(DEVICE, non_blocking=True)
        mask = mask.to(DEVICE, non_blocking=True)
        series_index = series_index.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(train):
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                mu, zero_logits = model(past, future, static, return_zero_logits=True)
                base_loss = masked_loss(target, mu, mask)
                zero_loss = zero_occurrence_loss(target, zero_logits, mask)
                if train and HIERARCHICAL_LOSS_WEIGHT > 0:
                    hier_loss = hierarchical_aux_loss(target, mu, mask, series_index)
                else:
                    hier_loss = torch.zeros((), device=target.device, dtype=base_loss.dtype)
                loss = base_loss + HIERARCHICAL_LOSS_WEIGHT * hier_loss + ZERO_LOSS_WEIGHT * zero_loss
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scheduler is not None and scaler.get_scale() >= old_scale:
                    scheduler.step()
        total_loss += float(loss.item()) * target.size(0)
        total_base_loss += float(base_loss.item()) * target.size(0)
        total_hier_loss += float(hier_loss.item()) * target.size(0)
        total_zero_loss += float(zero_loss.item()) * target.size(0)
        total_rows += target.size(0)
    denominator = max(total_rows, 1)
    return {'loss': total_loss / denominator, 'base_loss': total_base_loss / denominator, 'hier_loss': total_hier_loss / denominator, 'zero_loss': total_zero_loss / denominator}

@torch.no_grad()
def predict_ds(model, dataset, batch_size=BATCH):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=USE_AMP)
    model.eval()
    preds, targets, masks = ([], [], [])
    for past, future, static, target, mask, _, _ in loader:
        past = past.to(DEVICE, non_blocking=True)
        future = future.to(DEVICE, non_blocking=True)
        static = static.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            mu = model(past, future, static)
        preds.append(mu.float().cpu().numpy())
        targets.append(target.numpy())
        masks.append(mask.numpy())
    return (np.concatenate(preds, axis=0), np.concatenate(targets, axis=0), np.concatenate(masks, axis=0))

## 4. Origins chung, scaler theo cutoff và manifest
Encoder ngắn dùng đúng phần suffix của encoder dài; target và future features giống hệt nhau.
Lag_364 và rolling vẫn được giữ: SEQ không phải giới hạn toàn bộ lịch sử thô mà feature từng sử dụng.
Scaler tính theo train từng cutoff và chuẩn hóa khi lấy mẫu, tránh nhân đôi panel cho mỗi window.

Manifest khóa code/data/config/environment. Giữ WINDOWS cố định khi resume; chạy lại sẽ bỏ qua các stage hoàn tất.
Nếu đổi seed grid/window/batch/cutoff, dùng OUT_DIR mới. Không tự thay batch khi OOM vì hierarchical loss phụ thuộc batch.


In [ ]:
assert len(MODEL_PAST_COLS) == 51 and len(FUT_COLS) == 10 and len(STATIC_COLS) == 5
if not SMOKE and N_SERIES == 30490:
    assert n_series == 30490, 'Full experiment requires all 30,490 series; use WINDOW_SMOKE=1 for local samples.'

# Recompute per-cutoff evaluator contexts explicitly. No calibration.
RAW_PRICE = PRICE_OBS
del PRICE, PRICE_REL, PROMO, SNAP, PRICE_AVAILABLE

def common_origins(cutoff):
    # Zero-based origin inclusive; next H targets end no later than cutoff-1.
    result = np.arange(MAX_SEQ-1, cutoff-H, dtype=np.int64)
    if MAX_TRAIN_ORIGINS > 0 and len(result) > MAX_TRAIN_ORIGINS:
        result = result[np.linspace(0, len(result)-1, MAX_TRAIN_ORIGINS).round().astype(int)]
    assert len(result) and result[0] >= MAX_SEQ-1 and result[-1]+H < cutoff
    return result.tolist()

def moments(values, cutoff):
    # Accumulate in float64, in series chunks to avoid a full panel copy.
    sums = np.zeros(values.shape[-1], np.float64)
    squares = np.zeros_like(sums)
    count = 0
    for start in range(0, len(values), 128):
        part = values[start:start+128, :cutoff].astype(np.float64)
        sums += part.sum(axis=(0,1))
        squares += np.square(part).sum(axis=(0,1))
        count += part.shape[0]*part.shape[1]
    mean = sums/count
    std = np.sqrt(np.maximum(squares/count-mean**2, 0))
    std[std < 1e-6] = 1
    return mean.astype(np.float32), std.astype(np.float32)

def cutoff_state(cutoff):
    base = moments(FEAT, cutoff)
    local = moments(LOCAL_STATE, cutoff)
    hierarchy = {level: moments(HIER_CONTEXT[level], cutoff) for level in HIER_LEVELS}
    revenue = (SALES[:, cutoff-28:cutoff].astype(np.float64)
               * np.nan_to_num(RAW_PRICE[:, cutoff-28:cutoff], nan=0).astype(np.float64)).sum(axis=1)
    weights = np.power(np.maximum(revenue, 0), SERIES_WEIGHT_POWER)
    weights = np.clip(weights/max(weights.mean(), 1e-9), .25, 4) if weights.sum()>0 else np.ones(n_series)
    contexts = build_contexts(GROUPS, SALES[:, :cutoff], RAW_PRICE[:, :cutoff])
    return {'base':base, 'local':local, 'hierarchy':hierarchy,
            'weights':weights.astype(np.float32), 'contexts':contexts}

class WindowDataset(Dataset):
    def __init__(self, seq, origins, state, training=False):
        self.seq, self.origins, self.state, self.training = seq, origins, state, training

    def __len__(self):
        return n_series*len(self.origins)

    def __getitem__(self, index):
        series, position = divmod(index, len(self.origins))
        origin = self.origins[position]
        sl = slice(origin-self.seq+1, origin+1)
        future_sl = slice(origin+1, origin+H+1)
        mean, std = self.state['base']
        past = (FEAT[series, sl][:, PAST_IDX]-mean[PAST_IDX])/std[PAST_IDX]
        future = (FEAT[series, future_sl][:, FUT_IDX]-mean[FUT_IDX])/std[FUT_IDX]
        mean, std = self.state['local']
        local = (LOCAL_STATE[series, sl].astype(np.float32)-mean)/std
        parent = []
        for level in HIER_LEVELS:
            mean, std = self.state['hierarchy'][level]
            parent.append((HIER_CONTEXT[level][GROUPS[level][series], sl]-mean)/std)
        past = np.concatenate([past, local]+parent, axis=1).astype(np.float32)
        target = SALES[series, future_sl]
        assert past.shape == (self.seq, 51) and future.shape == (H,10) and target.shape == (H,)
        weight = float(self.state['weights'][series]) if self.training else 1.
        return (torch.from_numpy(np.ascontiguousarray(past)),
                torch.from_numpy(np.ascontiguousarray(future)),
                torch.tensor(STATIC[series],dtype=torch.long),
                torch.from_numpy(np.ascontiguousarray(target)),
                torch.full((H,),weight,dtype=torch.float32),
                torch.tensor(series),torch.tensor(origin))

class SharedOriginSampler:
    def __init__(self, n_origins, seed):
        self.n_origins, self.seed, self.epoch = n_origins, seed, 0

    def __len__(self):
        return self.n_origins*math.ceil(n_series/BATCH)

    def __iter__(self):
        rng = np.random.default_rng(self.seed+self.epoch)
        self.epoch += 1
        for position in rng.permutation(self.n_origins):
            order = rng.permutation(n_series)
            for start in range(0, n_series, BATCH):
                yield [int(s)*self.n_origins+int(position) for s in order[start:start+BATCH]]

def new_model(seed):
    seed_everything(seed)
    return TFTLite(51,10,STATIC_CARD,d_model=D_MODEL,heads=HEADS,
                   dropout=DROPOUT,horizon=H).to(DEVICE)

def train_loader(dataset, seed):
    return DataLoader(dataset,batch_sampler=SharedOriginSampler(len(dataset.origins),seed),
                      num_workers=0,pin_memory=USE_AMP)

all_cutoffs = sorted(set([INNER_END, INNER_VAL_END]+CONFIRM_CUTOFFS+[c-H for c in CONFIRM_CUTOFFS]))
origin_rows = [{'cutoff':c, 'origin_index':o, 'last_observed_day':o+1,
                'target_start_day':o+2,'target_end_day':o+H+1}
               for c in all_cutoffs for o in common_origins(c)]
origin_frame = pd.DataFrame(origin_rows)
assert (origin_frame.target_end_day <= origin_frame.cutoff).all()

def array_digest(values):
    digest = hashlib.sha256()
    for start in range(0,len(values),128):
        digest.update(np.ascontiguousarray(values[start:start+128]).tobytes())
    return digest.hexdigest()

manifest = {'version':1,'source_sha256':SOURCE_HASH,'smoke':SMOKE,'windows':WINDOWS,'seeds':SEEDS,
    'n_series':n_series,'series_sha256':hashlib.sha256(json.dumps(series_ids).encode()).hexdigest(),
    'sales_sha256':array_digest(SALES),'features_sha256':array_digest(FEAT),
    'price_sha256':array_digest(RAW_PRICE),
    'inner_cutoff':INNER_END,'refit_cutoff':INNER_VAL_END,'outer_end':OUTER_END,
    'confirmation_cutoffs':CONFIRM_CUTOFFS,'max_epochs':MAX_EPOCHS,'patience':PATIENCE,
    'max_origins':MAX_TRAIN_ORIGINS,'batch':BATCH,'d_model':D_MODEL,'heads':HEADS,
    'lr':LR,'dropout':DROPOUT,'weight_decay':WEIGHT_DECAY,'loss':LOSS,'tweedie_p':TWEEDIE_P,
    'hier_loss_weight':HIERARCHICAL_LOSS_WEIGHT,'zero_loss_weight':ZERO_LOSS_WEIGHT,
    'series_weight_power':SERIES_WEIGHT_POWER,'past_features':MODEL_PAST_COLS,'future_features':FUT_COLS,
    'torch':torch.__version__,'numpy':np.__version__,'device':str(DEVICE),
    'origins_sha256':hashlib.sha256(origin_frame.to_csv(index=False).encode()).hexdigest()}
manifest_path = OUT_DIR/'experiment_manifest.json'
if manifest_path.exists():
    if json.loads(manifest_path.read_text(encoding='utf-8')) != manifest:
        raise ValueError('Resume manifest mismatch. Use a new WINDOW_OUT_DIR for changed experiments.')
else:
    manifest_path.write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
runtime = {'torch':torch.__version__,'device':str(DEVICE),
           'gpu_name':torch.cuda.get_device_name(0) if USE_AMP else None,
           'amp_init_scale':1024.,'requested_run_seeds':RUN_SEEDS}
(OUT_DIR/'latest_runtime.json').write_text(json.dumps(runtime,indent=2),encoding='utf-8')
origin_frame.to_csv(OUT_DIR/'shared_origins.csv',index=False)
pd.DataFrame([{'kind':'past','feature':x} for x in MODEL_PAST_COLS]
             +[{'kind':'future','feature':x} for x in FUT_COLS]).to_csv(OUT_DIR/'feature_manifest.csv',index=False)
print('Shared origins per cutoff:',origin_frame.groupby('cutoff').size().to_dict())



## 5. Preflight
Kiểm tra target chung, suffix encoder, shape và một bước backward thật trên cửa sổ dài nhất. Chọn batch chung trước khi chạy thí nghiệm. OOM: giảm WINDOW_BATCH rồi dùng OUT_DIR mới.

In [ ]:
inner_state = cutoff_state(INNER_END)
outer_state = cutoff_state(INNER_VAL_END)
# Check exact target equality and encoder suffix equality across all windows.
datasets = [WindowDataset(seq,common_origins(INNER_END),inner_state,True) for seq in WINDOWS]
reference = datasets[-1][0]
for seq, dataset in zip(WINDOWS,datasets):
    sample = dataset[0]
    assert torch.equal(sample[3],reference[3])
    assert torch.equal(sample[0],reference[0][-seq:])
    assert torch.equal(sample[1],reference[1])
    assert sample[6].item()+H < INNER_END

# One real training step at largest window, using the SAME batch size as every run.
probe_model = new_model(SEEDS[0])
probe_loader = train_loader(datasets[-1],SEEDS[0])
probe_opt = torch.optim.AdamW(probe_model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
probe_scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP,init_scale=1024.)
if USE_AMP:
    torch.cuda.reset_peak_memory_stats()
probe_stats = run_epoch(probe_model,[next(iter(probe_loader))],optimizer=probe_opt,scaler=probe_scaler)
assert np.isfinite(probe_stats['loss'])
print('Preflight passed:',probe_stats,'| parameters:',sum(p.numel() for p in probe_model.parameters()))
if USE_AMP:
    print('Largest-window peak allocated GB:',torch.cuda.max_memory_allocated()/2**30)
del probe_model,probe_loader,probe_opt,probe_scaler,datasets,reference
gc.collect()
if USE_AMP:
    torch.cuda.empty_cache()



## 6. Huấn luyện và resume
Mỗi seed/window có stage chọn epoch và refit độc lập. Checkpoint không được chọn trên outer validation.
LR OneCycle dùng cùng ngân sách MAX_EPOCHS; refit chỉ chạy prefix số epoch đã chọn.
Không lưu model của tất cả lượt để tiết kiệm disk; lưu predictions và metric. Stage đang dở sẽ chạy lại từ đầu.
Mỗi lượt hoàn tất ghi status cuối cùng; stage lỗi được ghi rõ và dừng để không âm thầm thiếu cấu hình.


In [ ]:
def atomic_json(path, payload):
    tmp = path.with_suffix('.tmp')
    tmp.write_text(json.dumps(payload,ensure_ascii=False,indent=2),encoding='utf-8')
    tmp.replace(path)

def fit_phase(seq, seed, cutoff, state, directory, fixed_epochs=None):
    directory.mkdir(parents=True,exist_ok=True)
    selection = fixed_epochs is None
    model = new_model(seed)
    dataset = WindowDataset(seq,common_origins(cutoff),state,True)
    loader = train_loader(dataset,seed)
    optimizer = torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    # Same LR trajectory for selection/refit, even when refit uses fewer epochs.
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer,max_lr=LR,
        total_steps=MAX_EPOCHS*len(loader),pct_start=.3,anneal_strategy='cos')
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP,init_scale=1024.)
    eval_ds = WindowDataset(seq,[cutoff-1],state)
    best, selected, stale = float('inf'), 0, 0
    history = []
    started = time.perf_counter()
    if USE_AMP:
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    for epoch in range(1,(MAX_EPOCHS if selection else fixed_epochs)+1):
        epoch_start = time.perf_counter()
        stats = run_epoch(model,loader,optimizer=optimizer,scaler=scaler,scheduler=scheduler)
        if not np.isfinite(stats['loss']):
            raise FloatingPointError('Nonfinite training loss')
        score = None
        if selection:
            pred, actual, _ = predict_ds(model,eval_ds)
            _, score = evaluate_wrmsse5(actual,pred,state['contexts'])
            if not np.isfinite(score):
                raise FloatingPointError('Nonfinite inner-validation metric')
            if score < best-1e-4:
                best,selected,stale = score,epoch,0
            else:
                stale += 1
        if USE_AMP:
            torch.cuda.synchronize()
        history.append({'seq':seq,'seed':seed,'cutoff':cutoff,'epoch':epoch,
                        **stats,'inner_wrmsse5':score,'seconds':time.perf_counter()-epoch_start})
        pd.DataFrame(history).to_csv(directory/'history.csv',index=False)
        print(f'seq={seq} seed={seed} cutoff={cutoff} epoch={epoch} loss={stats["loss"]:.4f} inner={score}',flush=True)
        if selection and stale>=PATIENCE:
            break
    elapsed = time.perf_counter()-started
    usage = {'seconds':elapsed,'epochs_run':len(history),
             'peak_allocated_gb':torch.cuda.max_memory_allocated()/2**30 if USE_AMP else None,
             'parameters':sum(p.numel() for p in model.parameters())}
    del loader,optimizer,scheduler,scaler
    return model,selected,usage

def evaluate_run(model,seq,seed,cutoff,state):
    dataset = WindowDataset(seq,[cutoff-1],state)
    pred,actual,_ = predict_ds(model,dataset)
    assert pred.shape == (n_series,H) and np.isfinite(pred).all() and (pred>=0).all()
    np.testing.assert_array_equal(actual,SALES[:,cutoff:cutoff+H])
    rows,overall = evaluate_wrmsse5(actual,pred,state['contexts'])
    metrics = [{'seq':seq,'seed':seed,'cutoff':cutoff,'level':r['level'],
                **{k:v for k,v in r.items() if k!='level'},'WRMSSE_5levels':overall} for r in rows]
    horizon = []
    for week in range(4):
        sl = slice(week*7,(week+1)*7)
        _,score = evaluate_wrmsse5(actual[:,sl],pred[:,sl],state['contexts'])
        horizon.append({'seq':seq,'seed':seed,'cutoff':cutoff,'forecast_week':week+1,'WRMSSE_5levels':score})
    zeros = (SALES[:,:cutoff]==0).mean(axis=1)
    buckets = pd.cut(zeros,[-.01,.25,.5,.75,.99,1.],labels=['<25%','25-50%','50-75%','75-99%','99-100%'])
    bucket_rows = []
    for label in buckets.categories:
        mask = buckets==label
        if mask.any():
            error = pred[mask]-actual[mask]
            denominator = max(float(actual[mask].sum()),1e-9)
            bucket_rows.append({'seq':seq,'seed':seed,'cutoff':cutoff,'zero_bucket':str(label),
                'n_series':int(mask.sum()),'MAE':float(np.abs(error).mean()),
                'WMAPE':float(np.abs(error).sum()/denominator),'Bias':float(error.sum()/denominator)})
    return pred,metrics,horizon,bucket_rows

def run_status_table():
    rows = []
    for path in sorted(RUN_DIR.glob('*/status.json')):
        rows.append(json.loads(path.read_text(encoding='utf-8')))
    pd.DataFrame(rows).to_csv(OUT_DIR/'run_status.csv',index=False)

# Stage-level resume: completed inner selection/refit stages are reused;
# interrupted stages restart from scratch, never reuse partially trained models.
for seed in RUN_SEEDS:
    for seq in WINDOWS:
        key = f'seq{seq}_seed{seed}'
        inner_dir = RUN_DIR/(key+'_inner')
        inner_dir.mkdir(exist_ok=True)
        selection_path = inner_dir/'selection.json'
        if selection_path.exists():
            selected = json.loads(selection_path.read_text())['selected_epoch']
        else:
            model,selected,usage = fit_phase(seq,seed,INNER_END,inner_state,inner_dir)
            del model
            atomic_json(selection_path,{'selected_epoch':selected,**usage})
        selection_usage = json.loads(selection_path.read_text())
        for cutoff in [INNER_VAL_END]+CONFIRM_CUTOFFS:
            directory = RUN_DIR/(key+f'_cutoff{cutoff}')
            directory.mkdir(exist_ok=True)
            status_path = directory/'status.json'
            if status_path.exists() and json.loads(status_path.read_text())['status']=='completed':
                print('Resume completed:',directory.name)
                continue
            # Confirmations choose epoch only before their own forecast cutoff.
            state = outer_state if cutoff==INNER_VAL_END else cutoff_state(cutoff)
            local_selected = selected
            try:
                atomic_json(status_path,{'seq':seq,'seed':seed,'cutoff':cutoff,'status':'running'})
                if cutoff!=INNER_VAL_END:
                    nested = cutoff-H
                    model,local_selected,confirmation_usage = fit_phase(seq,seed,nested,cutoff_state(nested),directory/'inner')
                    del model
                model,_,usage = fit_phase(seq,seed,cutoff,state,directory,fixed_epochs=local_selected)
                pred,metrics,horizon,buckets = evaluate_run(model,seq,seed,cutoff,state)
                np.savez_compressed(directory/'predictions.npz',ids=np.array(series_ids),y_pred=pred,
                                    y_true=SALES[:,cutoff:cutoff+H])
                pd.DataFrame(metrics).to_csv(directory/'metrics.csv',index=False)
                pd.DataFrame(horizon).to_csv(directory/'horizon.csv',index=False)
                pd.DataFrame(buckets).to_csv(directory/'zero_buckets.csv',index=False)
                atomic_json(status_path,{'seq':seq,'seed':seed,'cutoff':cutoff,'status':'completed',
                    'selected_epoch':local_selected,
                    'inner_seconds':selection_usage['seconds'] if cutoff==INNER_VAL_END else confirmation_usage['seconds'],
                    **usage})
                del model,pred
            except Exception as exc:
                atomic_json(status_path,{'seq':seq,'seed':seed,'cutoff':cutoff,'status':'failed',
                                        'error':f'{type(exc).__name__}: {exc}'})
                raise
            finally:
                run_status_table()
                gc.collect()
                if USE_AMP:
                    torch.cuda.empty_cache()



## 7. Đóng góp, chi phí và kết luận
Improvement% = 100 × (WRMSSE56 − WRMSSEseq) / WRMSSE56; dương là tốt hơn 56 ngày.
So sánh paired seeds, từng cấp, zero buckets và tuần dự báo. Error bars là standard deviation giữa seed, không phải CI.
Không mặc nhiên lấy điểm thấp nhất làm bằng chứng ưu thế: cần xét độ ổn định và chi phí. Khi chênh lệch nhỏ/không ổn định, cân nhắc cửa sổ ngắn.
Additional origins là kiểm tra độ nhạy thời gian; lựa chọn cuối cần được xác nhận bằng test chưa dùng.


In [ ]:
def collect(filename):
    frames = []
    for path in sorted(RUN_DIR.glob('*/status.json')):
        status = json.loads(path.read_text())
        if status['status']=='completed':
            frames.append(pd.read_csv(path.parent/filename))
    if not frames:
        raise RuntimeError('No completed runs')
    return pd.concat(frames,ignore_index=True)

metrics = collect('metrics.csv')
metrics.to_csv(OUT_DIR/'validation_metrics.csv',index=False)
collect('horizon.csv').to_csv(OUT_DIR/'horizon_metrics.csv',index=False)
collect('zero_buckets.csv').to_csv(OUT_DIR/'zero_bucket_metrics.csv',index=False)
statuses = pd.read_csv(OUT_DIR/'run_status.csv')
statuses['total_training_seconds'] = statuses.seconds+statuses.inner_seconds
statuses.to_csv(OUT_DIR/'resource_usage.csv',index=False)
statuses[['seq','seed','cutoff','selected_epoch']].to_csv(OUT_DIR/'selected_epochs.csv',index=False)
histories = [pd.read_csv(p).assign(stage=p.parent.name) for p in RUN_DIR.glob('*/history.csv')]
pd.concat(histories,ignore_index=True).to_csv(OUT_DIR/'training_history.csv',index=False)
scores = metrics[metrics.level.eq('L1')][['seq','seed','cutoff','WRMSSE_5levels']].copy()
reference = scores[scores.seq.eq(56)][['seed','cutoff','WRMSSE_5levels']].rename(columns={'WRMSSE_5levels':'reference_56'})
paired = scores.merge(reference,on=['seed','cutoff'],validate='many_to_one')
paired['delta_vs_56'] = paired.WRMSSE_5levels-paired.reference_56
paired['improvement_pct'] = 100*(paired.reference_56-paired.WRMSSE_5levels)/paired.reference_56
paired['wins_vs_56'] = paired.delta_vs_56 < 0
paired.to_csv(OUT_DIR/'paired_comparisons_vs_56.csv',index=False)
main = paired[paired.cutoff.eq(INNER_VAL_END)]
summary = main.groupby('seq').agg(mean_wrmsse5=('WRMSSE_5levels','mean'),
    std_wrmsse5=('WRMSSE_5levels','std'),mean_improvement_pct=('improvement_pct','mean'),
    wins_vs_56=('wins_vs_56','sum'),n_seeds=('seed','nunique')).reset_index()
l12 = metrics[(metrics.cutoff==INNER_VAL_END)&metrics.level.eq('L12')].groupby('seq').WRMSSE.mean()
summary['mean_l12_wrmsse'] = summary.seq.map(l12)
usage = statuses[statuses.cutoff.eq(INNER_VAL_END)].groupby('seq').agg(
    mean_refit_seconds=('seconds','mean'),mean_selected_epochs=('selected_epoch','mean'),
    mean_total_training_seconds=('total_training_seconds','mean'),
    peak_allocated_gb=('peak_allocated_gb','max'))
summary = summary.merge(usage,on='seq')
summary.to_csv(OUT_DIR/'window_summary.csv',index=False)
levels = metrics[metrics.cutoff.eq(INNER_VAL_END)].groupby(['seq','level']).WRMSSE.agg(['mean','std'])
levels.to_csv(OUT_DIR/'level_summary.csv')
print(summary.round(4).to_string(index=False))
fig,axes = plt.subplots(1,2,figsize=(12,4))
axes[0].errorbar(summary.seq,summary.mean_wrmsse5,yerr=summary.std_wrmsse5.fillna(0),marker='o',capsize=4)
axes[0].set(xlabel='Encoder days',ylabel='Validation WRMSSE-5',title='Mean ± seed standard deviation')
axes[1].plot(summary.seq,summary.mean_refit_seconds,marker='o')
axes[1].set(xlabel='Encoder days',ylabel='Seconds',title='Mean refit training time')
fig.tight_layout();fig.savefig(OUT_DIR/'window_accuracy_cost.png',dpi=140);plt.show()
fig,ax = plt.subplots(figsize=(9,5))
for level in LEVELS:
    part = levels.xs(level,level='level')['mean']
    ax.plot(part.index,part.values,marker='o',label=level)
ax.set(xlabel='Encoder days',ylabel='WRMSSE',title='Validation error by hierarchy level')
ax.legend();fig.tight_layout();fig.savefig(OUT_DIR/'window_level_metrics.png',dpi=140);plt.show()
complete = len(main)==len(WINDOWS)*len(SEEDS)
best = summary.sort_values('mean_wrmsse5').iloc[0]
report = ['# TFT encoder-window experiment',
    '**SMOKE: pipeline verification, not model-selection evidence.**' if SMOKE else 'Full experiment; compare matched seeds and forecast origins.',
    f'Completed main grid: {complete}. Outer validation: d_1886-d_1913; BU without calibration.',
    '```text'+chr(10)+summary.round(5).to_string(index=False)+chr(10)+'```',
    f'Lowest observed mean WRMSSE-5: SEQ={int(best.seq)}. This is a candidate, not automatic proof of superiority.',
    'Use paired deltas, per-level tradeoffs, seed stability and additional origins before freezing the choice. Seed standard deviation is not a confidence interval.',
    'The common feature bank retains lag_364/rolling history even for short encoders. This experiment measures input sequence length conditional on that bank.',
    'Inner validation selects epoch; independent refit predicts outer validation. Confirmations have their own nested epoch selection. No competition test targets are loaded.',
    'Additional origins are temporal sensitivity checks, not a fresh untouched holdout. Shorter windows may be preferred when differences are practically small.',
    'Resume checks code/data/config hashes. Interrupted stages restart; completed stages are reused.']
(OUT_DIR/'window_length_report.md').write_text((chr(10)*2).join(report),encoding='utf-8')
print('Saved:',safe_str(OUT_DIR.resolve()))
